# Image Forgery Detection — Exploratory Analysis

Use this notebook to:
- Browse sample images and their ground-truth masks
- Visualise augmented outputs
- Inspect model predictions on single images
- Plot training curves from TensorBoard logs

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve() / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

## 1 — Load a sample forged image and its mask

In [ ]:
from data import ForgeryDataset, get_val_transforms

ds = ForgeryDataset('data/raw', split='train', transform=None, return_mask=True)
print(f'Dataset size: {len(ds)}')

if len(ds) > 0:
    img, mask, label = ds[0]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img)
    axes[0].set_title('Image')
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title(f'Mask  (label={label})')
    plt.tight_layout()
    plt.show()

## 2 — Inspect augmentation pipeline

In [ ]:
from data import get_train_transforms
import torch

transform = get_train_transforms(image_size=256)

if len(ds) > 0:
    raw_img, raw_mask, _ = ds[0]
    aug = transform(image=np.array(raw_img), mask=np.array(raw_mask))
    aug_img = aug['image'].permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(raw_img)
    axes[0].set_title('Original')
    axes[1].imshow((aug_img - aug_img.min()) / (aug_img.max() - aug_img.min()))
    axes[1].set_title('Augmented')
    plt.show()

## 3 — Model architecture summary

In [ ]:
import torch
from models import ForgeryDetector

model = ForgeryDetector(backbone='resnet50', pretrained=False)
x = torch.randn(1, 3, 256, 256)
logits, mask = model(x)
print(f'logits shape : {logits.shape}')
print(f'mask shape   : {mask.shape}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params : {total_params:,}')